In [1]:
import os
os.chdir(r'c:\Users\leopa\Documents\python codes\New folder\Information_retrieval\notebooks')
print(f"Current working directory: {os.getcwd()}")

Current working directory: c:\Users\leopa\Documents\python codes\New folder\Information_retrieval\notebooks


# Custom Model A: TF-IDF Vectorization & Cosine Similarity

In this notebook, we:
1. Reload the preprocessed reviews and setup the same Train/Test split.
2. Vectorize the *test* set and *train/query* set using `TfidfVectorizer`.
3. Compute Cosine Similarities between queries and test locations.
4. Evaluate the ranked results using Level 1 and Level 2 Error metrics.

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data & Train/Test Split (Same Seed as Baseline)

In [3]:
# Load prepared data
df_reviews = pd.read_csv('prepared_reviews.csv')
df_places = pd.read_csv('../data/Tripadvisor.csv', low_memory=False)

eval_cols = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places = df_places[eval_cols].copy()

df_merged = pd.merge(df_reviews, df_places, left_on='idplace', right_on='id', how='inner')

# Split
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy()
test_df = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

### 2. Evaluation Functions

### 2. Evaluation Functions

Same type-aware evaluation logic as in the BM25 baseline notebook.

In [4]:
def eval_level_1(query_typeR, sorted_test_typeR_list):
    """Level 1 error: rank of first match on typeR (H/R/A/AP). Error = rank index."""
    if pd.isna(query_typeR) or query_typeR not in sorted_test_typeR_list:
        return None
    for i, t in enumerate(sorted_test_typeR_list):
        if t == query_typeR:
            return i
    return None

def extract_subcategories(row):
    """
    Type-aware subcategory extraction (FIX: avoids spurious cross-type matches).

    Previously all subcategory columns were pooled into one flat set, which
    caused a hotel's priceRange to 'match' a restaurant's priceRange even though
    they are completely different venue types.  Now we only compare attributes
    that are semantically relevant for the place's own typeR.

    Per assignment spec:
      H  (Hotel)             -> priceRange
      R  (Restaurant)        -> restaurantType, restaurantTypeCuisine
      A / AP (Attraction)    -> activiteSubType
    """
    cats = set()
    type_r = str(row.get('typeR', '')).strip()
    if type_r == 'H':
        val = row.get('priceRange', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    elif type_r == 'R':
        for col in ['restaurantType', 'restaurantTypeCuisine']:
            val = row.get(col, '')
            if pd.notna(val) and str(val).strip():
                for p in str(val).split(','):
                    cats.add(p.strip().lower())
    elif type_r in ('A', 'AP'):
        val = row.get('activiteSubType', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    return cats

test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]
test_df['subcategories'] = test_subcats_list

def eval_level_2(query_subcats, sorted_test_indices, test_subcats_list):
    """Level 2 error: rank of first test place sharing at least one subcategory."""
    if not query_subcats:
        return None
    for i, test_idx in enumerate(sorted_test_indices):
        if query_subcats.intersection(test_subcats_list[test_idx]):
            return i
    return None


### 3. Model Pipeline: TF-IDF Vectorization

### 3. Model Pipeline: TF-IDF Vectorization

**FIX (Data Leakage):** Previously the `TfidfVectorizer` was `.fit_transform`'d on the **test** corpus, then `.transform`'d on the train (queries). This means the vocabulary and IDF weights were derived from test-set statistics, leaking information about the retrieval corpus into the query representation.

Correct approach: **fit on train**, transform both.

In [5]:
# CORRECTED: Fit on TRAIN set, transform BOTH sets.
# This ensures query representations use no information from the test corpus.
vectorizer = TfidfVectorizer()

train_texts = train_df['top_100_words'].fillna('').tolist()
test_texts  = test_df['top_100_words'].fillna('').tolist()

# fit_transform on TRAIN (queries)
train_vectors = vectorizer.fit_transform(train_texts)
# transform TEST (corpus)  — uses train vocabulary/IDF
test_vectors  = vectorizer.transform(test_texts)

print(f"Vocabulary size (from train): {len(vectorizer.vocabulary_)}")
print(f"Vectorized {train_vectors.shape[0]} Queries and {test_vectors.shape[0]} Test Docs.")


Vocabulary size (from train): 4755
Vectorized 917 Queries and 918 Test Docs.


### 4. Running the Benchmark

In [6]:
lvl1_errors = []
lvl2_errors = []

start_time = time.time()
print(f"Evaluating {len(train_df)} queries...")

# Compute cosine sim for all at once for speed
similarities = cosine_similarity(train_vectors, test_vectors)
test_type_array = test_df['typeR'].values

for i in range(len(train_df)):
    row = train_df.iloc[i]
    
    # Get similarities for query `i`
    sims = similarities[i]
    # Sort indices descending
    ranked_indices = np.argsort(sims)[::-1]
    test_typeR_list = test_type_array[ranked_indices].tolist()
    
    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None:
        lvl1_errors.append(err_1)
        
    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, ranked_indices, test_subcats_list)
    if err_2 is not None:
        lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 30)
print(f"Model A (TF-IDF + Cosine Sim) Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f}")
print(f"Model A (TF-IDF + Cosine Sim) Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f}")

Evaluating 917 queries...
Evaluation done in 0.07s
------------------------------
Model A (TF-IDF + Cosine Sim) Average Ranking Error Level 1: 0.68
Model A (TF-IDF + Cosine Sim) Average Ranking Error Level 2: 4.74
